## This notebook demonstrates a toy example of the `XGBOptClfWarm` class.

Note: Warm-starting is most beneficial when hyperparameters are transferred from another experiment on a similar dataset. 

For example, you could run the XGBOptClfWarm class on Dataset A, discover (some) good hyperparameters, and then use those as the starting point when training on Dataset B, provided Dataset B is similar in its structure.

For illustrative purposes, you can inspect a toy example of utilizing the aforementioned class on the same dataset.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from sklearn.datasets import make_classification
from src.xgb_opt_clf import *
from sklearn.model_selection import train_test_split
import logging
import pandas as pd
from src.helper_functions import *


In [5]:
# Generating a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2, random_state=42)

# Split into Train (for optimization/CV) and Val (for the final performance test)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [7]:
from sklearn.model_selection import train_test_split

# Split first — test set never seen by nested CV
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = XGBOptClf(n_trials=100)

# Run nested CV on dev only
nested_cv = nested_cv_score(clf, X_dev, y_dev, n_outer=5)

# Extract best params from dev experiment
best_fold_idx = nested_cv['scores'].argmax()
best_params   = nested_cv['best_params'][best_fold_idx]

# Warm-start on dev, evaluate on held-out test
clf_warm = XGBOptClfWarm(n_trials=100, initial_params=best_params)
clf_warm.fit(X_dev, y_dev)

In [8]:
results = clf_warm.eval(X_dev, y_dev, X_test, y_test)

In [9]:
report_performance(clf_warm.eval(X_dev=X_dev, X_test= X_test, y_test=y_test, y_dev=y_dev))

Threshold used:   0.3530

========== DEV_REPORT ==========
OVERALL ACCURACY: 0.9725
AUC SCORE:        0.9965
-----------------------------------
              precision  recall  f1-score  support
0                0.9797  0.9650    0.9723    400.0
1                0.9655  0.9800    0.9727    400.0
macro avg        0.9726  0.9725    0.9725    800.0
weighted avg     0.9726  0.9725    0.9725    800.0

========== TEST_REPORT ==========
OVERALL ACCURACY: 0.9000
AUC SCORE:        0.9689
-----------------------------------
              precision  recall  f1-score  support
0                0.8922    0.91     0.901    100.0
1                0.9082    0.89     0.899    100.0
macro avg        0.9002    0.90     0.900    200.0
weighted avg     0.9002    0.90     0.900    200.0
